### Preamble

In [ ]:
%load_ext autoreload
%autoreload 2

import os, pickle
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from traffic_scheduling.network.basics import generate_instance, uniform

# make sure the results directory exists
os.makedirs('results', exist_ok=True)

from datetime import datetime
now = lambda: datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Integer programming benchmark for network problems

We evaluate how fast the integer programming gap falls over time for crossing time scheduling problems in networks of increasing sizes and increasing total numbers of vehicles in the system.
We generate the a table showing the average gap after a minute of solving, averaged over a couple of random instances.

In [22]:
N_bench = 5       # number of instances per experiment
timelimit = 60     # maximum integer programming time per instance
F = uniform(0, 1) # shared route arrival distribution
exp = pd.DataFrame([
    [1, 2, 25],
    [1, 2, 50],
    [3, 4, 25],
    [3, 4, 50],
    [5, 5, 25],
    [5, 5, 50],
    [10, 10, 25],
    [10, 10, 50],
], columns=['net_m', 'net_n', 'n_r']) # n_r is number of arrivals per route

# compute experiment name for cache filename
exp['name'] = exp.apply(lambda row: f"{row['net_n']}x{row['net_m']}_{row['n_r']}", axis=1)
# compute total number of vehicles in system
exp['N'] = exp.apply(lambda row: row['n_r'] * (row['net_m'] + row['net_n']), axis=1)
# format network size string
exp['net'] = exp.apply(lambda row: f'${row['net_m']} \\times {row['net_n']}$', axis=1)

In [ ]:
def bench(name, instances, timelimit=60):
    options = { 'recordprogress': True, 'timelimit': timelimit }
    schedules  = []
    for s in tqdm(instances):
        schedules.append(s.solve(**options))
        
    with open(f"results/milp-bench-{name}.pkl", "wb") as file:
        pickle.dump(schedules, file)
    # compute average gap
    res = np.array([s.opt.gap for s in instances])
    return res.mean(), res.std()

for i, row in exp.iterrows():
    print(f"{now()} experiment {i}: {row['net_m']} x {row['net_n']}, N = {row['N']}")
    R = row['net_m'] + row['net_n']
    instances = [
        generate_instance(F, n=[row['n_r']]*R, net_m=row['net_m'], net_n=row['net_n'])
        for _ in range(N_bench)
    ]
    mean, std = bench(row['name'], instances, timelimit=timelimit)
    exp.loc[i, 'gap'] = mean
    exp.loc[i, 'gap_std'] = std

print(f"{now()} Done.")
os.system('notify-send "traffic-scheduling" "Benchmark done"');

2025-11-09 21:42:02 experiment 0: 1 x 2, N = 75


100%|██████████| 5/5 [05:00<00:00, 60.05s/it]


2025-11-09 21:47:02 experiment 1: 1 x 2, N = 150


100%|██████████| 5/5 [05:00<00:00, 60.15s/it]


2025-11-09 21:52:03 experiment 2: 3 x 4, N = 175


...

2025-11-09 22:12:16 experiment 6: 10 x 10, N = 500


100%|██████████| 5/5 [05:07<00:00, 61.46s/it]


2025-11-09 22:17:23 experiment 7: 10 x 10, N = 1000


100%|██████████| 5/5 [05:27<00:00, 65.45s/it]

2025-11-09 22:22:51 Done.


Produce LaTeX table for report.

In [29]:
exp

,net_m,net_n,n_r,name,N,net,gap,gap_std
0,1,2,25,2x1_25,75,$1 \times 2$,16.297674,0.830418
1,1,2,50,2x1_50,150,$1 \times 2$,24.770103,1.125638
2,3,4,25,4x3_25,175,$3 \times 4$,16.119093,0.452416
3,3,4,50,4x3_50,350,$3 \times 4$,28.957552,1.115679
4,5,5,25,5x5_25,250,$5 \times 5$,14.874434,0.480839
5,5,5,50,5x5_50,500,$5 \times 5$,34.327680,3.441578
6,10,10,25,10x10_25,500,$10 \times 10$,20.941523,9.509564
7,10,10,50,10x10_50,1000,$10 \times 10$,33.087352,4.821643


In [32]:
exp['gap'] = exp['gap'] / 100
exp['gap_std'] = exp['gap_std'] / 100

In [33]:
exp

,net_m,net_n,n_r,name,N,net,gap,gap_std
0,1,2,25,2x1_25,75,$1 \times 2$,16.297674,0.830418
1,1,2,50,2x1_50,150,$1 \times 2$,24.770103,1.125638
2,3,4,25,4x3_25,175,$3 \times 4$,16.119093,0.452416
3,3,4,50,4x3_50,350,$3 \times 4$,28.957552,1.115679
4,5,5,25,5x5_25,250,$5 \times 5$,14.874434,0.480839
5,5,5,50,5x5_50,500,$5 \times 5$,34.327680,3.441578
6,10,10,25,10x10_25,500,$10 \times 10$,20.941523,9.509564
7,10,10,50,10x10_50,1000,$10 \times 10$,33.087352,4.821643


In [34]:
tab = exp[['n_r', 'net', 'N', 'gap', 'gap_std']].copy()
tab['gap_txt'] = tab.apply(lambda row: f"{row['gap']:.2f}\\% \\pm {row['gap_std']:.2f}", axis=1)
tab.drop(columns=['gap', 'gap_std'], inplace=True)
tab.rename(columns={
    'n_r':  r'$n_r$',
    'net': 'network size',
    'gap_txt': 'final gap',
}, inplace=True)

latex_table = tab.to_latex(
    column_format='cccc', index=False,
    escape=False, float_format="%.2f")

# write table for use in report
with open('results/milp-bench.tex', 'w') as f:
    f.write(latex_table)

latex_table

'\\begin{tabular}{cccc}\n\\toprule\n$n_r$ & network size & N & final gap \\\\\n\\midrule\n25 & $1 \\times 2$ & 75 & 16.30\\% \\pm 0.83 \\\\\n50 & $1 \\times 2$ & 150 & 24.77\\% \\pm 1.13 \\\\\n25 & $3 \\times 4$ & 175 & 16.12\\% \\pm 0.45 \\\\\n50 & $3 \\times 4$ & 350 & 28.96\\% \\pm 1.12 \\\\\n25 & $5 \\times 5$ & 250 & 14.87\\% \\pm 0.48 \\\\\n50 & $5 \\times 5$ & 500 & 34.33\\% \\pm 3.44 \\\\\n25 & $10 \\times 10$ & 500 & 20.94\\% \\pm 9.51 \\\\\n50 & $10 \\times 10$ & 1000 & 33.09\\% \\pm 4.82 \\\\\n\\bottomrule\n\\end{tabular}\n'

Progress plot, showing gap over time.

In [28]:
def plot(name):
    with open(f"results/milp-bench-{name}.pkl", "rb") as file:
        schedules = pickle.load(file)
    plt.figure(figsize=(8,5))
    for s in schedules:
        plt.plot(s.progress["time"], s.progress["gap_percent"], linewidth=1.0, color='k', linestyle='--')
    plt.xlabel("Time (s)", fontsize=12)
    plt.ylabel(r"Gap ($\%$)", fontsize=12)
    plt.ylim(bottom=-1)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(f"figures/milp-bench-{name}.pdf")
    plt.show()